In [ ]:
"""
기상 피처 생성 (precipitation_mm, is_weather_alert, is_freezing)

data/daejeon_asos.csv: 대전 ASOS 관측소(지점번호 133) 시간 단위 관측
(2024-10-01 ~ 2026-07-14, 15,624시간, 결측 없이 1시간 간격 유지).
인코딩이 cp949(euc-kr)이므로 반드시 encoding="cp949"로 읽어야 함.

핵심 제약: ASOS 관측소는 대전에 1개(단일 지점)뿐이라 구간별로 다른 값을
줄 수 없다. 즉 이 피처는 segment_key와 무관하게 시간(timestamp)에만
의존하는 시 전체 공통값이며, 우리 90개 구간 모두가 같은 날씨 값을 받는다.

precipitation_mm:
  ASOS "강수량(mm)" 컬럼을 그대로 사용. 결측(89%)은 강수가 없어 기록을
  생략한 것이므로 0으로 채운다(ASOS 관례).

기준값 결정 과정 (중요):
  처음엔 KMA 공식 주의보 기준(강수 30mm/h, 3시간누적 60mm, 풍속 14m/s,
  3시간신적설 5cm)으로 시도했으나 발동 시간이 8시간(0.05%)뿐이라 학습
  피처로 쓰기엔 너무 희소했다. network 팀이 이미 인용한 한국교통연구원
  (2019) 강수-속도 관계 연구 기준(강수 30mm/h, 기온 0도 미만=결빙, 적설
  3cm 이상)으로 바꿔보니 "기온<0" 조건 하나가 전체 발동의 98%(2003/2039
  시간, 12.82%)를 차지하는 걸 확인했다 - 즉 원인이 전혀 다른 두 리스크
  (결빙 vs 호우/폭설)가 하나의 플래그에 뭉쳐 있었다.
  그래서 두 컬럼으로 분리한다:
    - is_freezing: 기온 0도 미만(결빙 리스크, 겨울철 다수 발생 - 12.82%)
    - is_weather_alert: 강수 30mm/h 이상 또는 적설(그 시점) 3cm 이상
      (호우/폭설 리스크, 희소 이벤트)
  두 기준 모두 한국교통연구원(2019) 문헌치를 그대로 사용 - network 팀
  산출물(segment_priority.csv의 기상 위험계수)과 동일 근거라 프로젝트
  전체 일관성이 유지된다. KMA의 3시간누적강수/풍속 조건은 이 문헌에
  없어 제외했다(풍속은 실제로 이 관측기간 내 14m/s를 한 번도 넘지 않아
  실효성도 없었음).

출력: output/features/weather_features.parquet
  timestamp(1시간 단위), precipitation_mm, is_weather_alert, is_freezing
"""

from pathlib import Path

import pandas as pd
import polars as pl

ASOS_PATH = "./data/daejeon_asos.csv"

RAIN_THRESHOLD = 30.0   # mm/h - 한국교통연구원(2019) 기준, network 팀과 동일
SNOW_THRESHOLD = 3.0    # cm(그 시점 적설) - 한국교통연구원(2019) 기준, network 팀과 동일
FREEZING_TEMP = 0.0     # °C 미만 - 결빙 리스크, 한국교통연구원(2019) 기준

OUTPUT_DIR = Path("./output/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ==================================================================
# 1. 원본 로드 + 정리
# ==================================================================

raw = pd.read_csv(ASOS_PATH, encoding="cp949")
raw = raw.rename(
    columns={
        "일시": "timestamp",
        "강수량(mm)": "precipitation_mm",
        "풍속(m/s)": "wind_ms",
        "적설(cm)": "snow_cm",
        "3시간신적설(cm)": "snow_3h_cm",
        "기온(°C)": "temp_c",
    }
)
raw["timestamp"] = pd.to_datetime(raw["timestamp"])
raw = raw[["timestamp", "precipitation_mm", "wind_ms", "snow_cm", "snow_3h_cm", "temp_c"]].sort_values("timestamp")

# 결측 처리: 강수량/적설/신적설은 "관측 안 됨=0"이 ASOS 관례
raw["precipitation_mm"] = raw["precipitation_mm"].fillna(0.0)
raw["snow_cm"] = raw["snow_cm"].fillna(0.0)
raw["snow_3h_cm"] = raw["snow_3h_cm"].fillna(0.0)

print(f"행 수: {len(raw)}")
print(f"기간: {raw['timestamp'].min()} ~ {raw['timestamp'].max()}")
print(f"시간 간격 균일 여부(1시간): {(raw['timestamp'].diff().dropna() == pd.Timedelta(hours=1)).all()}")
raw.describe()

In [ ]:
# ==================================================================
# 2. is_freezing / is_weather_alert 산출 (컬럼 분리)
# ==================================================================
# is_freezing: 기온 0도 미만 (결빙 리스크) - 겨울철에 흔하게 발생
# is_weather_alert: 강수 30mm/h 이상 또는 적설(그 시점) 3cm 이상 (호우/폭설, 희소)

raw["is_freezing"] = raw["temp_c"] < FREEZING_TEMP
raw["is_weather_alert"] = (raw["precipitation_mm"] >= RAIN_THRESHOLD) | (raw["snow_cm"] >= SNOW_THRESHOLD)

print(f"is_freezing=True: {raw['is_freezing'].sum()} / {len(raw)} ({raw['is_freezing'].mean():.2%})")
print(f"is_weather_alert=True: {raw['is_weather_alert'].sum()} / {len(raw)} ({raw['is_weather_alert'].mean():.2%})")
print()
print("is_weather_alert 조건별 발동 횟수:")
print("  강수 30mm/h+:", (raw["precipitation_mm"] >= RAIN_THRESHOLD).sum())
print("  적설(그 시점) 3cm+:", (raw["snow_cm"] >= SNOW_THRESHOLD).sum())
print()
print("월별 is_freezing 발생 시간 수 (상위 10개월):")
print(
    raw.assign(month=raw["timestamp"].dt.to_period("M"))
    .groupby("month")["is_freezing"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print()
print("두 조건 동시 발생(결빙+호우/폭설) 시간 수:", (raw["is_freezing"] & raw["is_weather_alert"]).sum())


In [ ]:
# ==================================================================
# 3. 저장
# ==================================================================

weather_features = raw[["timestamp", "precipitation_mm", "is_weather_alert", "is_freezing"]].copy()

pl.from_pandas(weather_features).write_parquet(OUTPUT_DIR / "weather_features.parquet")

print(f"저장 완료: {(OUTPUT_DIR / 'weather_features.parquet').resolve()}")
print(f"shape: {weather_features.shape}")
weather_features.head()


In [ ]:
"""
사용법 요약 (후속 XGBoost 피처 매트릭스 조립 시)

- key: timestamp만 있으면 됨(segment_key 없음 - 시 전체 단일 관측소라
  모든 구간에 동일 적용).
- weather_features는 1시간 단위, 우리 피처 매트릭스(10분 또는 5분 단위)의
  timestamp를 시(hour) 단위로 내림(floor)한 뒤 join해야 함:

    feature_df.with_columns(
        pl.col("timestamp").dt.truncate("1h").alias("weather_hour")
    ).join(
        weather_features.rename({"timestamp": "weather_hour"}),
        on="weather_hour",
        how="left",
    )

주의:
  - is_weather_alert / is_freezing 모두 실제 기상청 특보 발령 이력이 아니라
    한국교통연구원(2019) 문헌 기준을 ASOS 관측값에 적용한 근사치.
    network 팀의 기상 위험계수와 동일 근거라 프로젝트 내 일관성은 유지됨.
  - is_freezing은 12.82%로 흔한 편(겨울철 다수), is_weather_alert는
    희소(호우/폭설 실제 발생 시간만).
  - 공간 해상도가 없음(대전 전체 단일 값) - 실제로는 지역별 강수 편차가
    있을 수 있으나 반영 불가.
"""
print("완료")
